# Feature Engineering – Practical Machine Learning Series

## Learning objectives
By the end of this notebook, you should be able to explain the concept, identify appropriate techniques, implement them with pandas/scikit-learn, and recognize common risks.

**Important:** Feature engineering must be learned from the training data only when a transformation learns statistics or target relationships. Fit transformations on training data and apply the fitted transformation to validation/test data.

# 02 – Numerical Feature Engineering

Numerical feature engineering creates useful numeric representations from measurements, counts, amounts, and continuous variables. The right operation depends on units, domain meaning, data distribution, and the model.

In [ ]:
import numpy as np
import pandas as pd

sales = pd.DataFrame({
    "unit_price": [100, 250, 80, 400, 150],
    "quantity": [2, 3, 5, 1, 4],
    "discount": [10, 25, 0, 40, 15],
    "current_month_spend": [220, 700, 380, 360, 570],
    "previous_month_spend": [200, 650, 400, 300, 500],
    "visits": [4, 10, 2, 8, 5],
    "days_active": [20, 25, 10, 30, 15]
})
display(sales)

## 1. Basic mathematical operations

Use addition/subtraction when values share compatible units or when a meaningful difference is needed. Use multiplication for meaningful combined quantities. Division creates rates or ratios; protect against zero denominators and check units.

| Technique | Example | When/why |
|---|---|---|
| Addition | `price + delivery_fee` | Total compatible costs |
| Subtraction | `current - previous` | Change or gap |
| Multiplication | `price * quantity` | Total transaction value |
| Division | `amount / count` | Average or per-unit value |
| Ratio | `debt / income` | Relative burden |
| Percentage | `part / total * 100` | Share of a whole |

In [ ]:
sales["gross_value"] = sales["unit_price"] * sales["quantity"]       # multiplication
sales["discounted_value"] = sales["gross_value"] - sales["discount"] # subtraction
sales["price_plus_discount"] = sales["unit_price"] + sales["discount"] # addition example
sales["value_per_visit"] = sales["gross_value"] / sales["visits"].replace(0, np.nan)
sales["discount_pct"] = 100 * sales["discount"] / sales["gross_value"].replace(0, np.nan)
display(sales[["gross_value", "discounted_value", "value_per_visit", "discount_pct"]])

## 2. Differences, absolute difference, and relative difference

- **Difference:** `A - B`; preserves direction (increase or decrease).
- **Absolute difference:** `abs(A - B)`; measures gap size without direction.
- **Relative difference:** `(A - B) / abs(B)`; expresses change relative to a baseline. Define what happens when the baseline is zero.
- **Rate feature:** amount per unit of time or exposure, e.g. claims per policy-year.

Use relative changes to compare entities with different baselines, but avoid division by zero and interpret negative baselines carefully.

In [ ]:
sales["spend_change"] = sales["current_month_spend"] - sales["previous_month_spend"]
sales["absolute_spend_gap"] = (
    sales["current_month_spend"] - sales["previous_month_spend"]
).abs()
sales["relative_spend_change"] = (
    sales["spend_change"] / sales["previous_month_spend"].replace(0, np.nan)
)
sales["visits_per_active_day"] = (
    sales["visits"] / sales["days_active"].replace(0, np.nan)
)
display(sales[["spend_change", "absolute_spend_gap", "relative_spend_change", "visits_per_active_day"]])

## 3. Aggregation features

Aggregations summarize multiple observations or related columns. Examples include customer total spend, average order value, maximum claim amount, or transaction count.

Use `groupby` to aggregate historical records. Ensure the aggregation uses only records available before the prediction timestamp; otherwise future information can leak into the feature.

In [ ]:
transactions = pd.DataFrame({
    "customer_id": [1, 1, 1, 2, 2, 3],
    "amount": [100, 250, 150, 80, 120, 500]
})
customer_summary = transactions.groupby("customer_id").agg(
    total_spend=("amount", "sum"),
    average_order=("amount", "mean"),
    max_order=("amount", "max"),
    order_count=("amount", "count")
).reset_index()
display(customer_summary)

## 4. Log, square-root, and power transformations

- **Log transformation:** `log1p(x)` is useful for non-negative, strongly right-skewed values and compresses large values. `log1p` handles zero but not values below -1.
- **Square-root transformation:** often useful for non-negative count-like data with moderate right skew.
- **Power transformation:** Box-Cox requires strictly positive values; Yeo-Johnson can handle zero and negative values. These estimate a transformation parameter from training data.

Transformations can improve linear-model behavior and reduce the influence of extreme magnitudes. They are not always needed for tree-based models. Inspect distributions and validate performance.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.preprocessing import PowerTransformer

skewed = pd.Series([0, 1, 2, 3, 5, 10, 25, 60, 150, 500], name="amount")
transformed = pd.DataFrame({
    "original": skewed,
    "log1p": np.log1p(skewed),
    "sqrt": np.sqrt(skewed)
})
display(transformed)

# Yeo-Johnson can handle zero and negative values; fit on training data in real projects.
values = np.array([-10, -2, 0, 1, 5, 20, 100]).reshape(-1, 1)
pt = PowerTransformer(method="yeo-johnson")
print("Yeo-Johnson example:", np.round(pt.fit_transform(values).ravel(), 3))

## 5. Normalized features

Normalization is used to put measurements on comparable scales.

- **Min-max scaling:** maps values to a chosen range, commonly 0–1; sensitive to outliers.
- **Standardization:** centers around mean 0 and standard deviation 1; does not guarantee a fixed range.
- **Robust scaling:** uses median and interquartile range; less sensitive to outliers.

These are scaling methods rather than business feature creation. Fit scalers on training data only. Scaling is important for distance-based and gradient-based models, but often unnecessary for tree models.

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

numeric = pd.DataFrame({
    "income": [20000, 30000, 45000, 50000, 300000],
    "age": [21, 25, 32, 40, 55]
})
for name, scaler in {
    "minmax": MinMaxScaler(),
    "standard": StandardScaler(),
    "robust": RobustScaler()
}.items():
    result = scaler.fit_transform(numeric)  # demonstration only; split first in real ML
    print(name, "\\n", np.round(result[:3], 2))